# Assessment 2: Machine learning and real-time streaming

## Stefan Garevski (33759839)

### Dataset reuse and train/stream split

In [1]:
#initial imports
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import (
    col,
    when,
    hour,
    to_timestamp,
    countDistinct,
    sum,
    round,
    substring,
    row_number,
    spark_partition_id,
    min,
    max,
    avg,
    count
)
from pyspark.sql.window import Window

#initiate spark session
spark = SparkSession.builder \
    .appName("Victorian Road Crash Analysis") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/21 21:14:40 WARN Utils: Your hostname, Stefans-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.118 instead (on interface en0)
26/09/21 21:14:40 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/opt/anaconda3/envs/ITO5202/lib/python3.13/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/09/21 21:14:40 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
#define schema of variables in dataset 
schema = StructType([
    StructField("ACCIDENT_NO", StringType(), True),
    StructField("ACCIDENT_DATE", StringType(), True),
    StructField("ACCIDENT_TIME", StringType(), True),
    StructField("ACCIDENT_TYPE", StringType(), True),
    StructField("DAY_OF_WEEK", StringType(), True),
    StructField("DCA_CODE", StringType(), True),
    StructField("DCA_CODE_DESCRIPTION", StringType(), True),
    StructField("LIGHT_CONDITION", StringType(), True),
    StructField("POLICE_ATTEND", StringType(), True),
    StructField("ROAD_GEOMETRY", StringType(), True),
    StructField("SEVERITY", StringType(), True),
    StructField("SPEED_ZONE", StringType(), True),
    StructField("RUN_OFFROAD", StringType(), True),
    StructField("ROAD_NAME", StringType(), True),
    StructField("ROAD_TYPE", StringType(), True),
    StructField("ROAD_ROUTE_1", StringType(), True),
    StructField("LGA_NAME", StringType(), True),
    StructField("DTP_REGION", StringType(), True),
    StructField("LATITUDE", DoubleType(), True),
    StructField("LONGITUDE", DoubleType(), True),
    StructField("VICGRID_X", DoubleType(), True),
    StructField("VICGRID_Y", DoubleType(), True),
    StructField("TOTAL_PERSONS", IntegerType(), True),
    StructField("INJ_OR_FATAL", IntegerType(), True),
    StructField("FATALITY", IntegerType(), True),
    StructField("SERIOUSINJURY", IntegerType(), True),
    StructField("OTHERINJURY", IntegerType(), True),
    StructField("NONINJURED", IntegerType(), True),
    StructField("MALES", IntegerType(), True),
    StructField("FEMALES", IntegerType(), True),
    StructField("BICYCLIST", IntegerType(), True),
    StructField("PASSENGER", IntegerType(), True),
    StructField("DRIVER", IntegerType(), True),
    StructField("PEDESTRIAN", IntegerType(), True),
    StructField("PILLION", IntegerType(), True),
    StructField("MOTORCYCLIST", IntegerType(), True),
    StructField("UNKNOWN", IntegerType(), True),
    StructField("PED_CYCLIST_5_12", IntegerType(), True),
    StructField("PED_CYCLIST_13_18", IntegerType(), True),
    StructField("OLD_PED_65_AND_OVER", IntegerType(), True),
    StructField("OLD_DRIVER_75_AND_OVER", IntegerType(), True),
    StructField("YOUNG_DRIVER_18_25", IntegerType(), True),
    StructField("NO_OF_VEHICLES", IntegerType(), True),
    StructField("HEAVYVEHICLE", IntegerType(), True),
    StructField("PASSENGERVEHICLE", IntegerType(), True),
    StructField("MOTORCYCLE", IntegerType(), True),
    StructField("PT_VEHICLE", IntegerType(), True),
    StructField("DEG_URBAN_NAME", StringType(), True),
    StructField("SRNS", StringType(), True),
    StructField("RMA", StringType(), True),
    StructField("DIVIDED", StringType(), True),
    StructField("STAT_DIV_NAME", StringType(), True)
])

In [3]:
#load in csv data
file_path = "../Assessment 1/vic_road_crash_data.csv"

crashes = spark.read \
    .option("header", True) \
    .schema(schema) \
    .csv(file_path)

In [5]:
# create split
train_df, stream_df = crashes.randomSplit([0.7, 0.3], seed=42)
total = crashes.count()

# check percentage split
print("Training:", train_df.count(), f"({train_df.count()/total:.2%})")
print("Streaming:", stream_df.count(), f"({stream_df.count()/total:.2%})")

# save as parquet
stream_df.write.mode("overwrite").parquet("data/stream_data.parquet")

Training: 140622 (70.19%)
Streaming: 59730 (29.81%)


26/09/21 21:15:15 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/09/21 21:15:15 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/09/21 21:15:15 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/09/21 21:15:16 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/09/21 21:15:16 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
                                                                                

# Part A: The ML pipeline (45%)

## 1. ML task selection

For Part A, I have selected Classification as the machine learning task. The rationale is that the VicRoads crash dataset contains an appropriate categorical target variable called `SEVERITY`, which represents the severity category of each recorded road crash.

The dataset contains a mix of categorical and numerical variables that can be used as features in the model. Categorical variables include `ACCIDENT_TYPE`, `DAY_OF_WEEK`, `LIGHT_CONDITION`, `ROAD_GEOMETRY`, `ROAD_TYPE`, `SPEED_ZONE`, `RUN_OFFROAD`, and `LGA_NAME`. Numerical variables include `TOTAL_PERSONS`, `INJ_OR_FATAL`, `FATALITY`, `SERIOUSINJURY`, `OTHERINJURY`, `NO_OF_VEHICLES`, etc. These variables describe the crash, road environment, location, vehicles, and people involved. Therefore these variables can provide meaningful information for predicting the crash severity.

Classification is the most suitable in this instance because the objective is to predict which severity category a crash belongs to rather than predict a continuous numerical quantity. The `SEVERITY` column provides an existing categorical label that can be used, so there is no requirement to set any type of threshold for this activity.

Something like a regression is less suitable because most of the numerical variables generally describe counts or measurements associated with a crash rather than a prediction whose output would be a continuous number. Also, the `SEVERITY` variable is categorical rather than continuous.

Clustering is also not really suitable because it is primarily intended for situations where there may be no suitable target variable and the goal is to discover previously unknown groups or patterns in the data. In this instance, classification provides a more direct way to use the Victorian crash dataset.

Therefore, I will be using classification with `SEVERITY` as the target variable and relevant crash, road, vehicle, location, and environmental attributes as predictive features.

## 2. Feature engineering

## 3. Model comparison

## 4. Evaluation requirements

## 5. Model persistence

# Part B: Streaming, Kafka and reflection

## 1. Streaming data simulation

## 2. Kafka producer

## 3. Spark structured streaming consumer

## 4. Windowed aggregation

## 5. Reflection section